# 🔍 Text Branch Training — Multimodal Fake News Detection
### Fine-tuning RoBERTa-base on Fakeddit (3-way classification)

**What this does:** Takes a news headline → classifies it as **Real**, **Misleading**, or **Manipulated**

**Runtime:** ~1-2 hours on T4 GPU

**Steps:**
1. ✅ Install dependencies
2. ✅ Check GPU
3. 📁 Upload your Fakeddit TSV files
4. 🏋️ Train the model
5. 📊 Evaluate on test set
6. 💾 Download the trained checkpoint


In [ ]:
# Install required packages
!pip install -q torch transformers scikit-learn pandas tqdm

In [ ]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️  NO GPU DETECTED! Go to Runtime → Change runtime type → T4 GPU")

## 📁 Upload Data
Upload your 3 Fakeddit TSV files below:
- `multimodal_train.tsv`
- `multimodal_validate.tsv`
- `multimodal_test_public.tsv`

**Option A:** Use the upload cell below
**Option B:** Mount Google Drive if files are already there


In [ ]:
import os

# ── OPTION A: Upload from your computer ──
# Uncomment the next 2 lines to upload files directly:
# from google.colab import files
# uploaded = files.upload()

# ── OPTION B: Mount Google Drive ──
# Uncomment the next 2 lines if files are in your Drive:
from google.colab import drive
drive.mount('/content/drive')

# Set your data path
# If using Option A (direct upload): DATA_DIR = "/content"
# If using Option B (Drive): update the path below to match your Drive folder
DATA_DIR = "/content/drive/MyDrive/fakeddit"  # ← CHANGE THIS to your actual path

os.makedirs(DATA_DIR, exist_ok=True)
print(f"Data directory: {DATA_DIR}")
print(f"Files found: {os.listdir(DATA_DIR) if os.path.exists(DATA_DIR) else 'Directory not found!'}")

In [ ]:
from dataclasses import dataclass
from typing import Optional

@dataclass
class TextBranchConfig:
    # Model
    model_name: str = "roberta-base"
    num_labels: int = 3  # 0=Real, 1=Misleading, 2=Manipulated
    classifier_dropout: float = 0.3
    freeze_embeddings: bool = False
    freeze_n_layers: int = 0

    # Tokenizer
    max_seq_length: int = 256

    # Training
    batch_size: int = 32
    num_epochs: int = 5
    learning_rate: float = 2e-5
    weight_decay: float = 0.01
    warmup_ratio: float = 0.1
    max_grad_norm: float = 1.0
    fp16: bool = True  # Mixed precision for T4

    # Data — UPDATE filenames if yours are different
    data_dir: str = DATA_DIR
    train_file: str = "multimodal_train.tsv"
    val_file: str = "multimodal_validate.tsv"
    test_file: str = "multimodal_test_public.tsv"
    text_column: str = "clean_title"
    label_column: str = "3_way_label"
    max_train_samples: Optional[int] = 50000   # Subset for faster training
    max_val_samples: Optional[int] = 5000
    max_test_samples: Optional[int] = 5000
    num_workers: int = 2

    # Output
    output_dir: str = "/content/checkpoints"
    seed: int = 42
    label_names: tuple = ("Real", "Misleading", "Manipulated")

    @property
    def train_path(self):
        return os.path.join(self.data_dir, self.train_file)
    @property
    def val_path(self):
        return os.path.join(self.data_dir, self.val_file)
    @property
    def test_path(self):
        return os.path.join(self.data_dir, self.test_file)

config = TextBranchConfig()
print("✅ Config loaded")
print(f"  Model: {config.model_name}")
print(f"  Epochs: {config.num_epochs}")
print(f"  Batch size: {config.batch_size}")
print(f"  Train samples: {config.max_train_samples}")

In [ ]:
import torch.nn as nn
from transformers import RobertaModel

class TextClassifier(nn.Module):
    """
    RoBERTa-base + 2-layer classification head.

    Architecture:
        RoBERTa → [CLS] token (768-dim) → Dropout → Linear(768→256) → ReLU → Linear(256→3)
    """

    def __init__(self, model_name="roberta-base", num_labels=3,
                 classifier_dropout=0.3, freeze_embeddings=False,
                 freeze_n_layers=0):
        super().__init__()
        self.num_labels = num_labels
        self.roberta = RobertaModel.from_pretrained(model_name)
        self.hidden_size = self.roberta.config.hidden_size  # 768

        if freeze_embeddings:
            for param in self.roberta.embeddings.parameters():
                param.requires_grad = False

        if freeze_n_layers > 0:
            for i in range(min(freeze_n_layers, len(self.roberta.encoder.layer))):
                for param in self.roberta.encoder.layer[i].parameters():
                    param.requires_grad = False

        self.classifier = nn.Sequential(
            nn.Dropout(classifier_dropout),
            nn.Linear(self.hidden_size, 256),
            nn.ReLU(),
            nn.Dropout(classifier_dropout / 2),
            nn.Linear(256, num_labels),
        )

        # Xavier initialization for classifier head
        for module in self.classifier:
            if isinstance(module, nn.Linear):
                nn.init.xavier_uniform_(module.weight)
                nn.init.zeros_(module.bias)

    def forward(self, input_ids, attention_mask, return_features=False):
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        cls_embedding = outputs.last_hidden_state[:, 0, :]  # [CLS] token
        logits = self.classifier(cls_embedding)

        result = {"logits": logits}
        if return_features:
            result["features"] = cls_embedding  # 768-dim for fusion layer
        return result

# Build model
model = TextClassifier(
    model_name=config.model_name,
    num_labels=config.num_labels,
    classifier_dropout=config.classifier_dropout,
)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"✅ Model loaded")
print(f"  Trainable: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)")

In [ ]:
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from transformers import RobertaTokenizer

class FakedditTextDataset(Dataset):
    def __init__(self, tsv_path, tokenizer, max_seq_length=256,
                 text_column="clean_title", label_column="3_way_label",
                 max_samples=None):
        self.tokenizer = tokenizer
        self.max_seq_length = max_seq_length

        print(f"Loading {tsv_path}...")
        self.df = pd.read_csv(tsv_path, sep="\t", usecols=[text_column, label_column])
        self.df = self.df.dropna(subset=[text_column, label_column])
        self.df = self.df[self.df[text_column].str.strip().str.len() > 0]
        self.df[label_column] = self.df[label_column].astype(int)

        if max_samples and len(self.df) > max_samples:
            self.df = self.df.sample(n=max_samples, random_state=42).reset_index(drop=True)

        self.texts = self.df[text_column].tolist()
        self.labels = self.df[label_column].tolist()

        # Print class distribution
        counts = self.df[label_column].value_counts().sort_index()
        label_map = {0: "Real", 1: "Misleading", 2: "Manipulated"}
        print(f"  ✅ Loaded {len(self.texts):,} samples")
        for lid, cnt in counts.items():
            print(f"    {lid} ({label_map.get(lid, '?')}): {cnt:,} ({100*cnt/len(self.texts):.1f}%)")

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            str(self.texts[idx]),
            max_length=self.max_seq_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long),
        }

In [ ]:
tokenizer = RobertaTokenizer.from_pretrained(config.model_name)

train_dataset = FakedditTextDataset(config.train_path, tokenizer,
    max_seq_length=config.max_seq_length, max_samples=config.max_train_samples)
val_dataset = FakedditTextDataset(config.val_path, tokenizer,
    max_seq_length=config.max_seq_length, max_samples=config.max_val_samples)
test_dataset = FakedditTextDataset(config.test_path, tokenizer,
    max_seq_length=config.max_seq_length, max_samples=config.max_test_samples)

train_loader = DataLoader(train_dataset, batch_size=config.batch_size,
    shuffle=True, num_workers=config.num_workers, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=config.batch_size,
    shuffle=False, num_workers=config.num_workers, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=config.batch_size,
    shuffle=False, num_workers=config.num_workers, pin_memory=True)

print(f"\n✅ DataLoaders ready")
print(f"  Train: {len(train_loader)} batches")
print(f"  Val:   {len(val_loader)} batches")
print(f"  Test:  {len(test_loader)} batches")

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

def compute_metrics(preds, labels):
    preds, labels = np.array(preds), np.array(labels)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro", zero_division=0),
        "f1_weighted": f1_score(labels, preds, average="weighted", zero_division=0),
    }

@torch.no_grad()
def evaluate(model, dataloader, device):
    model.eval()
    all_preds, all_labels = [], []
    for batch in dataloader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        preds = torch.argmax(outputs["logits"], dim=-1).cpu().tolist()
        all_preds.extend(preds)
        all_labels.extend(batch["labels"].tolist())
    return all_preds, all_labels

print("✅ Evaluation functions ready")

## 🏋️ Training
This cell runs the full training loop. Expected time: **~1-2 hours on T4 GPU**.

Features:
- Mixed precision (FP16) for faster training
- Differential learning rate (5× higher for classifier head)
- Class weights to handle Misleading class imbalance (only 2.5% of data)
- Best model saved automatically based on validation F1


In [ ]:
import time
from torch.optim import AdamW
from torch.cuda.amp import GradScaler, autocast
from transformers import get_linear_schedule_with_warmup
from tqdm import tqdm
import random

# Set seeds for reproducibility
random.seed(config.seed)
np.random.seed(config.seed)
torch.manual_seed(config.seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(config.seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on: {device}")

model.to(device)

# Optimizer with differential learning rates
no_decay = ["bias", "LayerNorm.weight", "LayerNorm.bias"]
optimizer_grouped_params = [
    {"params": [p for n, p in model.roberta.named_parameters()
                if not any(nd in n for nd in no_decay) and p.requires_grad],
     "lr": config.learning_rate, "weight_decay": config.weight_decay},
    {"params": [p for n, p in model.roberta.named_parameters()
                if any(nd in n for nd in no_decay) and p.requires_grad],
     "lr": config.learning_rate, "weight_decay": 0.0},
    {"params": model.classifier.parameters(),
     "lr": config.learning_rate * 5, "weight_decay": config.weight_decay},
]
optimizer = AdamW(optimizer_grouped_params)

total_steps = len(train_loader) * config.num_epochs
warmup_steps = int(total_steps * config.warmup_ratio)
scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)
scaler = GradScaler(enabled=config.fp16)

# Class weights to handle imbalance (Misleading is only 2.5% of data)
class_weights = torch.tensor([1.0, 15.0, 0.7]).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)

os.makedirs(config.output_dir, exist_ok=True)
best_f1 = 0.0

print(f"Total steps: {total_steps}, Warmup: {warmup_steps}")
print(f"Starting training for {config.num_epochs} epochs...\n")

start_time = time.time()

for epoch in range(config.num_epochs):
    model.train()
    total_loss = 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{config.num_epochs}")

    for batch in pbar:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        with autocast(enabled=config.fp16):
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = criterion(outputs["logits"], labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), config.max_grad_norm)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        optimizer.zero_grad()

        total_loss += loss.item()
        pbar.set_postfix({"loss": f"{loss.item():.4f}", "lr": f"{scheduler.get_last_lr()[0]:.2e}"})

    avg_loss = total_loss / len(train_loader)

    # Validation
    val_preds, val_labels = evaluate(model, val_loader, device)
    val_metrics = compute_metrics(val_preds, val_labels)

    print(f"\nEpoch {epoch+1}: loss={avg_loss:.4f} | "
          f"val_acc={val_metrics['accuracy']:.4f} | "
          f"val_f1={val_metrics['f1_macro']:.4f}")

    # Save best model
    if val_metrics["f1_macro"] > best_f1:
        best_f1 = val_metrics["f1_macro"]
        local_path = os.path.join(config.output_dir, "best_model.pt")
        torch.save({
            "epoch": epoch + 1,
            "model_state_dict": model.state_dict(),
            "best_f1": best_f1,
            "config": {
                "model_name": config.model_name,
                "num_labels": config.num_labels,
                "classifier_dropout": config.classifier_dropout,
                "max_seq_length": config.max_seq_length,
            },
        }, local_path)
        size_mb = os.path.getsize(local_path) / 1e6
        print(f"  New best model saved! F1={best_f1:.4f} ({size_mb:.0f} MB)")

    # Classification report
    print(classification_report(val_labels, val_preds,
          target_names=list(config.label_names), digits=4, zero_division=0))

total_time = time.time() - start_time
print(f"\n{'='*50}")
print(f"Training complete in {total_time/60:.1f} minutes")
print(f"Best validation F1: {best_f1:.4f}")
print(f"{'='*50}")

## 📊 Final Test Evaluation
Loading the best checkpoint and evaluating on the held-out test set.


In [ ]:
# Load best checkpoint
checkpoint = torch.load(os.path.join(config.output_dir, "best_model.pt"), map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
print(f"Loaded best model from epoch {checkpoint['epoch']} (val F1={checkpoint['best_f1']:.4f})")

# Test evaluation
test_preds, test_labels = evaluate(model, test_loader, device)
test_metrics = compute_metrics(test_preds, test_labels)

print(f"\n{'='*50}")
print(f"  FINAL TEST RESULTS")
print(f"{'='*50}")
print(f"  Accuracy:      {test_metrics['accuracy']:.4f}")
print(f"  F1 (macro):    {test_metrics['f1_macro']:.4f}")
print(f"  F1 (weighted): {test_metrics['f1_weighted']:.4f}")
print()
print(classification_report(test_labels, test_preds,
      target_names=list(config.label_names), digits=4, zero_division=0))

# Confusion matrix
cm = confusion_matrix(test_labels, test_preds)
print("Confusion Matrix:")
header = f"{'':15s}{'Real':>12s}{'Misleading':>12s}{'Manipulated':>12s}"
print(header)
for i, name in enumerate(config.label_names):
    row = f"{name:15s}"
    for j in range(len(config.label_names)):
        row += f"{cm[i][j]:12d}"
    print(row)

## 🔮 Try It Out!
Test the trained model on some sample headlines.


In [ ]:
@torch.no_grad()
def predict_text(text, model, tokenizer, device):
    model.eval()
    encoding = tokenizer(text, max_length=256, padding="max_length",
                         truncation=True, return_tensors="pt")
    input_ids = encoding["input_ids"].to(device)
    attention_mask = encoding["attention_mask"].to(device)

    outputs = model(input_ids=input_ids, attention_mask=attention_mask, return_features=True)
    probs = torch.softmax(outputs["logits"], dim=-1).cpu().numpy()[0]
    pred = int(np.argmax(probs))

    label_map = {0: "✅ Real", 1: "⚠️ Misleading", 2: "🚨 Manipulated"}
    print(f"\nText: \"{text[:100]}\"")
    print(f"  → {label_map[pred]} (confidence: {probs[pred]:.1%})")
    print(f"    Real: {probs[0]:.1%}  |  Misleading: {probs[1]:.1%}  |  Manipulated: {probs[2]:.1%}")

# Test with sample headlines
test_headlines = [
    "Scientists confirm new vaccine is 95% effective in clinical trials",
    "SHOCKING: Government hiding alien bodies in secret underground base!!!",
    "Breaking: Celebrity caught in deepfake scandal targeting political opponents",
    "Local community garden project brings neighbors together for annual harvest",
    "You won't BELIEVE what this politician said about the economy",
]

print("=" * 60)
print("  INFERENCE DEMO")
print("=" * 60)
for headline in test_headlines:
    predict_text(headline, model, tokenizer, device)

## 💾 Download Trained Model
Run the cell below to download `best_model.pt` to your computer.
Save it to `text_branch/checkpoints/best_model.pt` in your project.


In [ ]:
from google.colab import files
checkpoint_path = os.path.join(config.output_dir, "best_model.pt")
print(f"Downloading: {checkpoint_path}")
print(f"File size: {os.path.getsize(checkpoint_path) / 1e6:.1f} MB")
files.download(checkpoint_path)
print("\n✅ Save this file to: text_branch/checkpoints/best_model.pt")